# NylC Diversity Panel postprocessing after AlphaFold2 modeling

Requires relaxed rank-1 ColabFold PDB models and their `scores` JSON files in `RAW_AF2_DIR` and `data/cleavage_site.csv`.
The notebook (1) renames chains and renumbers residues to match NylC X-ray structure 5y0m, (2) computes descriptors
(pLDDT, pTM/ipTM, pI, buried surface area, fpocket, active-site loops, key residues, activity) and (3) merges them into one CSV for machine learning.
Finally it compares the new table to the legacy table.

Megan M. Wisniewski and Jerry M. Parks..

In [ ]:
import os, sys, glob, json, math, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Paths (override with environment variables) -------------------------------------
DATA_DIR    = Path(os.environ.get('NYLC_DATA_DIR', 'data'))              # inputs (read-only)
RESULTS_DIR = Path(os.environ.get('NYLC_RESULTS_DIR', 'results'))        # everything written by this notebook
RAW_AF2_DIR = Path(os.environ.get('NYLC_RAW_AF2_DIR', 'af2_models/raw')) # relaxed ColabFold PDB + JSON files
LEGACY_DIR  = Path(os.environ.get('NYLC_LEGACY_DIR', 'legacy'))          # legacy outputs, for consistency checks
N_JOBS      = int(os.environ.get('NYLC_N_JOBS', '8'))
FORCE       = os.environ.get('NYLC_FORCE', '0') == '1'                   # recompute even if outputs exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR.resolve(), '\nresults:', RESULTS_DIR.resolve())
from datetime import datetime
from Bio.PDB import PDBParser
from Bio.SeqUtils.IsoelectricPoint import IsoelectricPoint as IP

TODAY = datetime.today().strftime('%Y%m%d')
FINAL_DIR   = RESULTS_DIR / 'final_af2_all'   # processed models
FPOCKET_DIR = RESULTS_DIR / 'fpocket'         # trimmed models + fpocket output
CSV_DIR     = RESULTS_DIR / 'features'        # one CSV per feature group
for d in (FINAL_DIR, FPOCKET_DIR, CSV_DIR):
    d.mkdir(exist_ok=True)
SCRIPTS = Path('scripts')
MSA_FILE = DATA_DIR / 'nylc_all.mafft.fasta'
CLEAVAGE_CSV = DATA_DIR / 'cleavage_site.csv'   ## provided; identical to the output of the prep notebook
THREE_TO_ONE = {'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
                'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}

In [ ]:
def parse_fasta(filename):
    """Return (headers, sequences). Handles blank lines; sequences are upper-cased."""
    header, sequence = [], []
    with open(filename) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                header.append(line[1:]); sequence.append([])
            else:
                sequence[-1].append(line.upper())
    return header, [''.join(s) for s in sequence]

names, seqs = parse_fasta(MSA_FILE)
enzyme_names = [n.split('/')[0].lower() for n in names]
assert enzyme_names[0] == 'nylc-gyaq' and len(set(enzyme_names)) == len(enzyme_names)
beta_start = dict(pd.read_csv(CLEAVAGE_CSV).values)
assert set(beta_start) == set(enzyme_names), 'cleavage_site.csv and MSA disagree'
print(len(enzyme_names), 'enzymes in the MSA; MSA length', len(seqs[0]))

def raw_files(enz):
    pdb = sorted(RAW_AF2_DIR.glob(f'{enz}_4x2_relaxed_rank_001_*.pdb'))
    js = sorted(RAW_AF2_DIR.glob(f'{enz}_4x2_scores_rank_001_*.json'))
    return pdb, js

missing = [e for e in enzyme_names if len(raw_files(e)[0]) != 1 or len(raw_files(e)[1]) != 1]
print('enzymes without exactly one relaxed rank-1 PDB and one JSON in', RAW_AF2_DIR, ':', missing)
assert not missing, 'Download/copy the AlphaFold2 models first (see README).' 

## Rename chains and renumber residues
AF2 numbers each chain from 1, and the alpha and beta chains have separate IDs (A-D alpha, E-H beta). `scripts/postprocess_af2_model.py` (PyMOL) aligns each
model to 5y0m, orders the chains as in 5y0m, gives each alpha/beta pair one chain ID and continues the beta-chain numbering from the alpha chain.
It also writes trimmed A+D models for fpocket. PyMOL runs in subprocesses because running it inside the notebook can kill the kernel.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def run_postprocess(enz):
    out = FINAL_DIR / f'{enz}.pdb'
    if out.exists() and (FPOCKET_DIR / f'{enz}_trimmed_ad.pdb').exists() and not FORCE:
        return enz, 0
    pdb = raw_files(enz)[0][0]
    cmd = [sys.executable, str(SCRIPTS / 'postprocess_af2_model.py'), '--model', str(pdb), '--beta_start', str(beta_start[enz]),
           '--output_dir', str(FINAL_DIR), '--output_dir_fpocket', str(FPOCKET_DIR), '--ref', str(DATA_DIR / '5y0m_tetramer.pdb')]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode: print(enz, r.stderr[-500:])
    return enz, r.returncode

with ThreadPoolExecutor(N_JOBS) as ex:
    res = list(ex.map(run_postprocess, enzyme_names))
failed = [e for e, rc in res if rc]
print('post-processed:', len(res) - len(failed), 'failed:', failed)
assert not failed

## Sequence and AlphaFold confidence descriptors
Chain lengths, isoelectric points, region-wise pLDDT (chain A; 100-column MSA windows, counted in MSA coordinates) and pTM/ipTM.

In [ ]:
def split_alpha_beta_seqs(df, seqs, col):
    """Split MSA rows at the cleavage column (alpha = columns < col, beta = columns >= col)."""
    df = df.copy()
    df['alpha_seq'] = [s[:col].replace('-', '') for s in seqs]
    df['beta_seq'] = [s[col:].replace('-', '') for s in seqs]      # FIX: legacy used s[col:-1], dropping the last MSA column
    df['num_res_alpha'] = df.alpha_seq.str.len(); df['num_res_beta'] = df.beta_seq.str.len()
    df['pi_alpha'] = [round(IP(s).pi(), 2) for s in df.alpha_seq]
    df['pi_beta'] = [round(IP(s).pi(), 2) for s in df.beta_seq]
    return df.drop(columns=['alpha_seq', 'beta_seq'])[['enzyme', 'num_res_alpha', 'num_res_beta', 'pi_alpha', 'pi_beta']]

def msa_to_residue_numbers(seq, msa_range):
    """1-based MSA column range (inclusive) -> 1-based residue numbers of the ungapped sequence."""
    nums, n = [], 0
    for col, aa in enumerate(seq, start=1):
        if aa != '-':
            n += 1
            if msa_range[0] <= col <= msa_range[1]:
                nums.append(n)
    return nums

def plddt_features(enzymes, seqs, regions, thresh=50):
    cols = {}
    for enz, seq in zip(enzymes, seqs):
        chain = PDBParser(QUIET=True).get_structure('m', FINAL_DIR / f'{enz}.pdb')[0]['A']
        row = {'enzyme': enz}
        for sel, rng in regions.items():
            vals = np.array([chain[n]['CA'].get_bfactor() for n in msa_to_residue_numbers(seq, rng) if n in chain and 'CA' in chain[n]])
            row[f'num_res_{sel}'] = len(vals)
            row[f'mean_plddt_{sel}'] = round(float(vals.mean()), 2) if len(vals) else np.nan
            row[f'num_res_plddt_lt_{thresh}_{sel}'] = int((vals < thresh).sum())
            row[f'frac_res_plddt_lt_{thresh}_{sel}'] = round((vals < thresh).sum() / len(vals), 3) if len(vals) else np.nan
        cols[enz] = row
    return pd.DataFrame(cols.values())

def ptm_features(enzymes):
    rows = []
    for enz in enzymes:
        d = json.load(open(raw_files(enz)[1][0]))
        rows.append({'enzyme': enz, 'ptm': d['ptm'], 'iptm': d['iptm']})
    return pd.DataFrame(rows)

cleavage_col = seqs[0].index('TTIS')      # MSA column of Thr267 in NylC-GYAQ (0-based)
df_seq = split_alpha_beta_seqs(pd.DataFrame({'enzyme': enzyme_names}), seqs, cleavage_col)
msa_len = len(seqs[0])
regions = {f'region_{i+1}': (i*100 + 1, min((i+1)*100, msa_len)) for i in range(math.ceil(msa_len/100))}
df_plddt = df_seq.merge(plddt_features(enzyme_names, seqs, regions), on='enzyme')
df_plddt.to_csv(CSV_DIR / 'nylc_all.plddt.csv', index=False, na_rep='NaN')
ptm_features(enzyme_names).to_csv(CSV_DIR / 'nylc_all.ptm_iptm.csv', index=False, na_rep='NaN')
print('Wrote nylc_all.plddt.csv and nylc_all.ptm_iptm.csv'); df_plddt.iloc[:3, :10]

## Buried surface area (A-B and A-D interfaces)
Shrake-Rupley SASA from Biopython (probe 1.4 A, 100 points); BSA = (SASA_A + SASA_other - SASA_dimer)/2. Takes several minutes.

In [ ]:
bsa_csv = CSV_DIR / 'nylc_all.bsa.csv'
if FORCE or not bsa_csv.exists():
    subprocess.run([sys.executable, str(SCRIPTS / 'bsa_descriptor_calculation.py'), '--pdb_dir', str(FINAL_DIR),
                    '--output_csv', str(bsa_csv), '--n_jobs', str(N_JOBS)], check=True, stdout=subprocess.DEVNULL)
pd.read_csv(bsa_csv).head()

## Active-site pocket descriptors (fpocket)
The ten residues before the cleavage site are removed (they can block the active site) and only chains A and D are kept (`*_trimmed_ad.pdb`).
fpocket is run on each model; descriptors are taken from the pocket whose alpha spheres are nearest to OG1 of the catalytic Thr in chain A.
Pockets depend strongly on side-chain conformations, so treat these descriptors with caution.

In [ ]:
fp_csv = CSV_DIR / 'nylc_all.fpocket.csv'
if FORCE or not fp_csv.exists():
    subprocess.run([sys.executable, str(SCRIPTS / 'fpocket_descriptors.py'), '--fpocket_dir', str(FPOCKET_DIR), '--cleavage_csv', str(CLEAVAGE_CSV),
                    '--output_csv', str(fp_csv), '--n_jobs', str(N_JOBS)], check=True, stdout=subprocess.DEVNULL)
pd.read_csv(fp_csv).head()

## Active-site loops
Loops are defined by NylC-GYAQ residue ranges and mapped to every sequence through the MSA.

In [ ]:
loop_selections = {'loop_1': (88, 92), 'loop_2': (146, 156), 'loop_3': (184, 200), 'loop_4': (220, 224),
                   'loop_5': (298, 304), 'loop_6': (320, 334), 'loop_7': (95, 99), 'loop_8': (129, 140)}
ref_seq = seqs[0]
loop_indices = {}
for loop_name, (start, end) in loop_selections.items():
    idx, count = [], 0
    for i, res in enumerate(ref_seq):
        if res != '-':
            count += 1
        if start <= count <= end:
            idx.append(i)
    loop_indices[loop_name] = idx

rows = []
for enz, seq in zip(enzyme_names, seqs):
    row = {'enzyme': enz}
    for loop_name, idx in loop_indices.items():
        loop_seq = ''.join(seq[i] for i in idx).replace('-', '')
        row[f'{loop_name}_len'] = len(loop_seq)
        row[f'{loop_name}_start'] = len(seq[:idx[0]].replace('-', '')) + 1
        row[f'{loop_name}_end'] = len(seq[:idx[-1] + 1].replace('-', ''))
        row[f'{loop_name}_seq'] = loop_seq
    rows.append(row)
df_loops = pd.DataFrame(rows)
df_loops.to_csv(CSV_DIR / 'nylc_all.loops.csv', index=False, na_rep='NaN'); df_loops.iloc[:3, :9]

## Key catalytic residues
Residue numbers (per sequence) equivalent to Tyr146, Asn219, Thr267 and Asp306 of NylC-GYAQ. `N/A` = the aligned position is a gap.
The legacy column `Thr219_equiv` was mislabeled and is called `Thr267_equiv` here.

In [ ]:
motifs = {'Tyr146_equiv': ('YDFS', 146), 'Asn219_equiv': ('NPVG', 219), 'Thr267_equiv': ('TTIS', 267), 'Asp306_equiv': ('DGDT', 306)}
ref_ungapped = ref_seq.replace('-', '')
cols = {}
for col, (motif, resnum) in motifs.items():
    pos = ref_seq.find(motif)
    assert pos != -1 and ref_ungapped.find(motif) + 1 == resnum, f'{motif} is not residue {resnum} of NylC-GYAQ'
    cols[col] = [sum(1 for r in s[:pos+1] if r != '-') if s[pos] != '-' else 'N/A' for s in seqs]
df_key = pd.DataFrame({'enzyme': enzyme_names, **cols})
df_key.to_csv(CSV_DIR / 'nylc_all.key_residues.csv', index=False); df_key.head(3)

## Enzyme activity
Mass-spec-based activity on nylon 6 and nylon 6,6 powder for the original diversity panel (1 = active, 0 = inactive); NaN = not tested.
NylC-GYAQ is kept (the legacy filter removed it by mistake); wild-type NylC has no model and is excluded.

In [ ]:
df_act = pd.read_csv(DATA_DIR / 'nylc_activity.csv')
df_act = df_act[df_act.enzyme.isin(enzyme_names)]
df_act.to_csv(CSV_DIR / 'nylc_all.activity.csv', index=False, na_rep='NaN'); print(len(df_act), 'enzymes with activity data')

## Merge all features

In [ ]:
files = ['plddt', 'ptm_iptm', 'bsa', 'fpocket', 'loops', 'key_residues', 'activity']
merged = None
for f in files:
    d = pd.read_csv(CSV_DIR / f'nylc_all.{f}.csv')
    merged = d if merged is None else pd.merge(merged, d, on='enzyme', how='outer')
assert merged.enzyme.is_unique and set(merged.enzyme) == set(enzyme_names)
out = RESULTS_DIR / f'nylc_all.merged_data_{TODAY}.csv'
merged.to_csv(out, index=False, na_rep='NaN')
print(merged.shape, '->', out)

## Consistency check against the legacy dataset (`nylc_all.merged_data_20250310.csv`)
Rows are matched by enzyme (the 24 extra picks are ignored). Numeric columns are compared with a tolerance; strings exactly.
`Thr219_equiv` in the legacy file corresponds to `Thr267_equiv`.

In [ ]:
leg = pd.read_csv(LEGACY_DIR / 'nylc_all.merged_data_20250310.csv').rename(columns={'Thr219_equiv': 'Thr267_equiv'})
leg = leg[leg.enzyme.isin(merged.enzyme)].set_index('enzyme').sort_index()
new = merged.set_index('enzyme').sort_index()
print('rows new/legacy-common:', len(new), len(leg), '| columns only in new:', sorted(set(new.columns) - set(leg.columns)),
      '| only in legacy:', sorted(set(leg.columns) - set(new.columns)))
report = []
for c in new.columns:
    if c not in leg.columns: continue
    a, b = new[c], leg[c]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        diff = ~((a - b).abs() <= 1e-2 + 1e-3 * b.abs()) & ~(a.isna() & b.isna())
    else:
        diff = (a.astype(str) != b.astype(str))
    if diff.any():
        report.append({'column': c, 'n_diff': int(diff.sum()), 'example_enzymes': ', '.join(new.index[diff][:5]),
                       'max_abs_diff': float((a - b).abs().max()) if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b) else np.nan})
report = pd.DataFrame(report)
report.to_csv(RESULTS_DIR / f'legacy_comparison_{TODAY}.csv', index=False)
print(len(new.columns) - len(report), 'of', len(new.columns), 'columns identical to legacy (within tolerance)')
report